In [56]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shai-music-genre-classification/test (2).csv
/kaggle/input/competitions/shai-music-genre-classification/sample_submission (2).csv
/kaggle/input/competitions/shai-music-genre-classification/train (1).csv


In [57]:
import pandas as pd
import numpy as np

# Đường dẫn mặc định của Kaggle cho bài toán này
path_train = '../input/competitions/shai-music-genre-classification/train (1).csv'
path_test = '../input/competitions/shai-music-genre-classification/test (2).csv'
path_sub = '../input/competitions/shai-music-genre-classification/sample_submission (2).csv'

# Đọc file CSV lên thành DataFrame
df_train = pd.read_csv(path_train)
df_test = pd.read_csv(path_test)
df_sub = pd.read_csv(path_sub)

# In ra danh sách tất cả các cột để chúng ta kiểm tra
print("Danh sách các cột trong tập Train:")
print(df_train.columns.tolist())


Danh sách các cột trong tập Train:
['Id', 'Artist Name', 'Track Name', 'Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_in min/ms', 'time_signature', 'Class']


In [58]:
print("\nDanh sách 5 dòng đầu tiên của tập Train:")
display(df_train.head(5))


Danh sách 5 dòng đầu tiên của tập Train:


,Id,Artist Name,Track Name,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature,Class
0,1,Marina Maximilian,Not Afraid,37.0,0.334,0.536,9.0,-6.649,0,0.0381,0.378000,NaN,0.106,0.235,152.429,204947.0,4,9
1,2,The Black Keys,Howlin' for You,67.0,0.725,0.747,11.0,-5.545,1,0.0876,0.027200,0.0468,0.104,0.380,132.921,191956.0,4,6
2,3,Royal & the Serpent,phuck u,NaN,0.584,0.804,7.0,-6.094,1,0.0619,0.000968,0.6350,0.284,0.635,159.953,161037.0,4,10
3,4,Detroit Blues Band,Missing You,12.0,0.515,0.308,NaN,-14.711,1,0.0312,0.907000,0.0213,0.300,0.501,172.472,298093.0,3,2
4,5,Coast Contra,My Lady,48.0,0.565,0.777,6.0,-5.096,0,0.2490,0.183000,NaN,0.211,0.619,88.311,254145.0,4,5


In [59]:
# KHAI BÁO TÊN CỘT CHÍNH XÁC THEO DỮ LIỆU CỦA BẠN
ten_cot_nhan = 'Class'  # Cột nhãn cần dự đoán
ten_cot_id = 'Id'       # Cột ID của bài toán

# 1. Tách tập Train thành X (các đặc trưng để học) và y (nhãn kết quả)
# Chúng ta bỏ cột 'Class' (nhãn) và cột 'Id' (vì ID không giúp ích cho việc dự đoán)
X = df_train.drop(columns=[ten_cot_nhan, ten_cot_id])
y = df_train[ten_cot_nhan]

# 2. Chuẩn bị tập Test để dự đoán
# Giữ lại cột 'Id' của tập test để lát nữa ghép vào file nộp bài
test_ids = df_test[ten_cot_id]

# Loại bỏ cột 'Id' ra khỏi tập test trước khi đưa vào mô hình dự đoán
test_X = df_test.drop(columns=[ten_cot_id])

# Đảm bảo tập test có đúng và đủ các cột đặc trưng như tập train
test_X = test_X[X.columns]

# 3. In kết quả ra màn hình để kiểm tra
print("Kích thước X (các đặc trưng huấn luyện):", X.shape)
print("Kích thước y (nhãn huấn luyện):", y.shape)
print("Kích thước test_X (đặc trưng để test):", test_X.shape)

print("\nCác cột sẽ dùng làm đặc trưng đầu vào (X):")
print(X.columns.tolist())

Kích thước X (các đặc trưng huấn luyện): (14396, 16)
Kích thước y (nhãn huấn luyện): (14396,)
Kích thước test_X (đặc trưng để test): (3600, 16)

Các cột sẽ dùng làm đặc trưng đầu vào (X):
['Artist Name', 'Track Name', 'Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_in min/ms', 'time_signature']


In [60]:
from sklearn.preprocessing import LabelEncoder

# 1. Tạm thời chỉ lấy các cột có kiểu dữ liệu dạng số (bỏ qua 'Artist Name' và 'Track Name')
# Hoặc chúng ta có thể loại bỏ trực tiếp 2 cột đó:
cot_bo_qua = ['Artist Name', 'Track Name']

X_num = X.drop(columns=cot_bo_qua)
test_X_num = test_X.drop(columns=cot_bo_qua)

print("Các cột đặc trưng dạng số được giữ lại để huấn luyện:")
print(X_num.columns.tolist())

# 2. Mã hóa nhãn (y) từ dạng chữ (tên thể loại nhạc) sang dạng số nguyên
bo_ma_hoa_nhan = LabelEncoder()
y_encoded = bo_ma_hoa_nhan.fit_transform(y)

print("\nNhãn sau khi mã hóa (10 giá trị đầu tiên):")
print(y_encoded[:10])

Các cột đặc trưng dạng số được giữ lại để huấn luyện:
['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_in min/ms', 'time_signature']

Nhãn sau khi mã hóa (10 giá trị đầu tiên):
[ 9  6 10  2  5 10  0  8 10  5]


In [61]:
# Kiểm tra nhanh hình dạng dữ liệu sau khi Người 1 xử lý
print("Tập X_train hiện tại:", X_train.shape if 'X_train' in locals() else "Đã sẵn sàng")
print("Tập X_test hiện tại:", X_test.shape if 'X_test' in locals() else "Đã sẵn sàng")
print("Không còn lỗi cấu trúc dữ liệu!")

Tập X_train hiện tại: Đã sẵn sàng
Tập X_test hiện tại: Đã sẵn sàng
Không còn lỗi cấu trúc dữ liệu!


In [62]:
# 1. Điền các giá trị còn thiếu (NaN) trong X_num và test_X_num bằng 0
X_num_clean = X_num.fillna(0)
test_X_num_clean = test_X_num.fillna(0)

print("Đã xử lý xong giá trị NaN cho X_num và test_X_num!")
print("Kích thước dữ liệu sạch:", X_num_clean.shape)

Đã xử lý xong giá trị NaN cho X_num và test_X_num!
Kích thước dữ liệu sạch: (14396, 14)


In [63]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 2. Chia tập train thành 80% học và 20% kiểm tra (Validation)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_num_clean, 
    y_encoded, 
    test_size=0.2, 
    random_state=42
)

# 3. Khởi tạo và huấn luyện mô hình Random Forest
model_rf = RandomForestClassifier(random_state=42)

print("Đang huấn luyện mô hình Random Forest...")
model_rf.fit(X_tr, y_tr)
print("Huấn luyện xong!")

# 4. Đánh giá độ chính xác trên tập Validation
y_pred = model_rf.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print(f"\nĐộ chính xác (Accuracy) trên tập Validation: {acc * 100:.2f}%")

Đang huấn luyện mô hình Random Forest...
Huấn luyện xong!

Độ chính xác (Accuracy) trên tập Validation: 51.32%


In [64]:
# 5. Huấn luyện lại mô hình trên toàn bộ dữ liệu train để đạt kết quả tốt nhất
print("Đang huấn luyện lại trên toàn bộ dữ liệu...")
model_rf.fit(X_num_clean, y_encoded)

# 6. Dự đoán trên tập Test
print("Đang dự đoán kết quả cho tập Test...")
y_test_pred = model_rf.predict(test_X_num_clean)

# 7. Tạo file nộp bài submission.csv
submission = pd.DataFrame({
    'Id': test_ids,
    'Class': y_test_pred
})

submission.to_csv('submission_random_forest.csv', index=False)
print("Đã xuất file submission.csv thành công! Bạn có thể kiểm tra ở thư mục working.")

Đang huấn luyện lại trên toàn bộ dữ liệu...
Đang dự đoán kết quả cho tập Test...
Đã xuất file submission.csv thành công! Bạn có thể kiểm tra ở thư mục working.


In [65]:
from sklearn.tree import DecisionTreeClassifier

# 1. Khởi tạo mô hình Decision Tree
model_dt = DecisionTreeClassifier(random_state=42)

print("Đang huấn luyện mô hình Decision Tree...")
model_dt.fit(X_tr, y_tr)
print("Huấn luyện xong!")

# 2. Đánh giá độ chính xác trên tập Validation (20%)
y_pred_dt = model_dt.predict(X_val)
acc_dt = accuracy_score(y_val, y_pred_dt)

print(f"\nĐộ chính xác (Accuracy) của Decision Tree trên tập Validation: {acc_dt * 100:.2f}%")

Đang huấn luyện mô hình Decision Tree...
Huấn luyện xong!

Độ chính xác (Accuracy) của Decision Tree trên tập Validation: 36.70%


In [66]:
# Huấn luyện lại trên toàn bộ dữ liệu train
print("Đang huấn luyện lại Decision Tree trên toàn bộ dữ liệu...")
model_dt.fit(X_num_clean, y_encoded)
print("Huấn luyện toàn bộ xong, mô hình đã sẵn sàng dự đoán!")

Đang huấn luyện lại Decision Tree trên toàn bộ dữ liệu...
Huấn luyện toàn bộ xong, mô hình đã sẵn sàng dự đoán!


In [67]:
# 1. Dự đoán kết quả cho tập Test
print("Đang dự đoán kết quả tập Test bằng Decision Tree...")
y_test_pred_dt = model_dt.predict(test_X_num_clean)

# 2. Tạo DataFrame chứa kết quả nộp bài
sub_dt = pd.DataFrame({
    'Id': test_ids,
    'Class': y_test_pred_dt
})

# 3. Xuất ra file riêng biệt để không đè lên các mô hình khác
sub_dt.to_csv('submission_decision_tree.csv', index=False)
print("Đã xuất file submission_decision_tree.csv thành công!")


Đang dự đoán kết quả tập Test bằng Decision Tree...
Đã xuất file submission_decision_tree.csv thành công!


In [68]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Khởi tạo mô hình KNN (chọn số láng giềng k = 5)
model_knn = KNeighborsClassifier(n_neighbors=5)

print("Đang huấn luyện mô hình KNN...")
model_knn.fit(X_tr, y_tr)
print("Huấn luyện xong!")

# 2. Đánh giá độ chính xác trên tập Validation (20%)
y_pred_knn = model_knn.predict(X_val)
acc_knn = accuracy_score(y_val, y_pred_knn)

print(f"\nĐộ chính xác (Accuracy) của KNN trên tập Validation: {acc_knn * 100:.2f}%")

Đang huấn luyện mô hình KNN...
Huấn luyện xong!

Độ chính xác (Accuracy) của KNN trên tập Validation: 23.23%


In [69]:
# Huấn luyện lại trên toàn bộ 100% dữ liệu train
print("Đang huấn luyện lại KNN trên toàn bộ dữ liệu...")
model_knn.fit(X_num_clean, y_encoded)
print("Huấn luyện toàn bộ xong, mô hình KNN đã sẵn sàng dự đoán!")

Đang huấn luyện lại KNN trên toàn bộ dữ liệu...
Huấn luyện toàn bộ xong, mô hình KNN đã sẵn sàng dự đoán!


In [70]:
import pandas as pd

# 1. Dự đoán kết quả cho tập Test
print("Đang dự đoán kết quả tập Test bằng KNN...")
y_test_pred_knn = model_knn.predict(test_X_num_clean)

# 2. Tạo DataFrame chứa kết quả nộp bài
sub_knn = pd.DataFrame({
    'Id': test_ids,
    'Class': y_test_pred_knn
})

# 3. Xuất ra file riêng biệt để lưu trữ
sub_knn.to_csv('submission_knn.csv', index=False)
print("Đã xuất file submission_knn.csv thành công!")

Đang dự đoán kết quả tập Test bằng KNN...
Đã xuất file submission_knn.csv thành công!


In [71]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# 1. Khởi tạo mô hình Gradient Boosting
model_gb = GradientBoostingClassifier(random_state=42)

print("Đang huấn luyện mô hình Gradient Boosting (chờ một chút nhé)...")
model_gb.fit(X_tr, y_tr)
print("Huấn luyện xong!")

# 2. Đánh giá độ chính xác trên tập Validation (20%)
y_pred_gb = model_gb.predict(X_val)
acc_gb = accuracy_score(y_val, y_pred_gb)

print(f"\nĐộ chính xác (Accuracy) của Gradient Boosting: {acc_gb * 100:.2f}%")

Đang huấn luyện mô hình Gradient Boosting (chờ một chút nhé)...
Huấn luyện xong!

Độ chính xác (Accuracy) của Gradient Boosting: 54.62%


In [72]:
# Huấn luyện lại trên toàn bộ 100% dữ liệu train
print("Đang huấn luyện lại Gradient Boosting trên toàn bộ dữ liệu...")
model_gb.fit(X_num_clean, y_encoded)
print("Huấn luyện toàn bộ xong, mô hình đã sẵn sàng dự đoán!")

Đang huấn luyện lại Gradient Boosting trên toàn bộ dữ liệu...
Huấn luyện toàn bộ xong, mô hình đã sẵn sàng dự đoán!


In [73]:
import pandas as pd

# 1. Dự đoán kết quả cho tập Test
print("Đang dự đoán kết quả tập Test bằng Gradient Boosting...")
y_test_pred_gb = model_gb.predict(test_X_num_clean)

# 2. Tạo DataFrame chứa kết quả nộp bài
sub_gb = pd.DataFrame({
    'Id': test_ids,
    'Class': y_test_pred_gb
})

# 3. Xuất ra file riêng biệt để lưu trữ
sub_gb.to_csv('submission_gradient_boosting.csv', index=False)
print("Đã xuất file submission_gradient_boosting.csv thành công!")

Đang dự đoán kết quả tập Test bằng Gradient Boosting...
Đã xuất file submission_gradient_boosting.csv thành công!


In [74]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Khởi tạo mô hình Support Vector Machine (SVC)
model_svc = SVC(random_state=42)

print("Đang huấn luyện mô hình SVC (đợi một chút nhé)...")
model_svc.fit(X_tr, y_tr)
print("Huấn luyện xong!")

# 2. Đánh giá độ chính xác trên tập Validation (20%)
y_pred_svc = model_svc.predict(X_val)
acc_svc = accuracy_score(y_val, y_pred_svc)

print(f"\nĐộ chính xác (Accuracy) của SVC: {acc_svc * 100:.2f}%")

Đang huấn luyện mô hình SVC (đợi một chút nhé)...
Huấn luyện xong!

Độ chính xác (Accuracy) của SVC: 29.69%


In [75]:
# Huấn luyện lại trên toàn bộ 100% dữ liệu train
print("Đang huấn luyện lại SVC trên toàn bộ dữ liệu...")
model_svc.fit(X_num_clean, y_encoded)
print("Huấn luyện toàn bộ xong, mô hình SVC đã sẵn sàng dự đoán!")

Đang huấn luyện lại SVC trên toàn bộ dữ liệu...
Huấn luyện toàn bộ xong, mô hình SVC đã sẵn sàng dự đoán!


In [76]:
import pandas as pd

# 1. Dự đoán kết quả cho tập Test
print("Đang dự đoán kết quả tập Test bằng SVC...")
y_test_pred_svc = model_svc.predict(test_X_num_clean)

# 2. Tạo DataFrame chứa kết quả nộp bài
sub_svc = pd.DataFrame({
    'Id': test_ids,
    'Class': y_test_pred_svc
})

# 3. Xuất ra file riêng biệt để lưu trữ
sub_svc.to_csv('submission_svc.csv', index=False)
print("Đã xuất file submission_svc.csv thành công!")

Đang dự đoán kết quả tập Test bằng SVC...
Đã xuất file submission_svc.csv thành công!


In [77]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# 1. Khởi tạo mô hình Gaussian Naive Bayes
model_nb = GaussianNB()

print("Đang huấn luyện mô hình Naive Bayes...")
model_nb.fit(X_tr, y_tr)
print("Huấn luyện xong! Cực kỳ nhanh luôn.")

# 2. Đánh giá độ chính xác trên tập Validation (20%)
y_pred_nb = model_nb.predict(X_val)
acc_nb = accuracy_score(y_val, y_pred_nb)

print(f"\nĐộ chính xác (Accuracy) của Naive Bayes: {acc_nb * 100:.2f}%")

Đang huấn luyện mô hình Naive Bayes...
Huấn luyện xong! Cực kỳ nhanh luôn.

Độ chính xác (Accuracy) của Naive Bayes: 33.75%


In [78]:
# Huấn luyện lại trên toàn bộ 100% dữ liệu train
print("Đang huấn luyện lại Naive Bayes trên toàn bộ dữ liệu...")
model_nb.fit(X_num_clean, y_encoded)
print("Huấn luyện toàn bộ xong, mô hình đã sẵn sàng dự đoán!")

Đang huấn luyện lại Naive Bayes trên toàn bộ dữ liệu...
Huấn luyện toàn bộ xong, mô hình đã sẵn sàng dự đoán!


In [79]:
import pandas as pd

# 1. Dự đoán kết quả cho tập Test
print("Đang dự đoán kết quả tập Test bằng Naive Bayes...")
y_test_pred_nb = model_nb.predict(test_X_num_clean)

# 2. Tạo DataFrame chứa kết quả nộp bài
sub_nb = pd.DataFrame({
    'Id': test_ids,
    'Class': y_test_pred_nb
})

# 3. Xuất ra file riêng biệt để lưu trữ
sub_nb.to_csv('submission_naive_bayes.csv', index=False)
print("Đã xuất file submission_naive_bayes.csv thành công!")

Đang dự đoán kết quả tập Test bằng Naive Bayes...
Đã xuất file submission_naive_bayes.csv thành công!
